In [7]:
import ipywidgets as ipyw
from IPython.display import display, clear_output
import os
import pandas as pd
import matplotlib.pyplot as plt


# Create output widget for plot
output = ipyw.Output()

# Initial setup
base_dir = "results/pdps/D/1_1"
estimators = {n.split("_")[1].split(".")[0] for n in os.listdir(base_dir)}
estimators_checkboxes = {est: ipyw.Checkbox(value=True, description=est) for est in estimators}

col_selection = ipyw.Select(value="X1", options=[f"X{i}" for i in range(1, 10)], rows=9, description="Column:")
configuration_selection = ipyw.Select(value="D", options=os.listdir("results/pdps/"), rows=9, description="Config:")

def get_scenario_options(config):
    path = f"results/pdps/{config}"
    if os.path.exists(path):
        return [f for f in os.listdir(path) if os.path.isdir(f"{path}/{f}")]
    return ["1_1"]

def update_scenario_options(change):
    new_scenarios = get_scenario_options(configuration_selection.value)
    old_value = scenario_selection.value
    scenario_selection.options = new_scenarios
    if new_scenarios and old_value in new_scenarios:
        scenario_selection.value = old_value
    else:
        scenario_selection.value = new_scenarios[0] if new_scenarios else "1_1"
    invalidate(None)

# Create scenario after config
scenarios = get_scenario_options(configuration_selection.value)
scenario_selection = ipyw.Select(value=scenarios[0] if scenarios else "1_1", 
                               options=scenarios or ["1_1"], 
                               rows=9, description="Scenario:")

def plot_data(column, configuration, scenario, selected_estimators):
    with output:
        output.clear_output(wait=True)
        plt.figure(figsize=(16,9))
        
        base_dir = f"results/pdps/{configuration}/{scenario}"
        
        for est in selected_estimators:
            try:
                df = pd.read_csv(f"{base_dir}/{column}_{est}.csv")
                plt.plot(df.index, df["mean_pdp"], label=est)
            except:
                pass
        
        try:
            plt.plot(df.index, df["ground_truth_pdp"], label="Ground Truth", color="black", ls="-.")
        except:
            pass
        
        plt.grid(alpha=0.3)
        plt.legend()
        plt.title(f"PDP: {column} | Config: {configuration}, Scenario: {scenario}")
        plt.axvline(x=len(df)*0.25, c="k", label="1st quartile", ls="--")
        plt.axvline(x=len(df)*0.75, c="k", label="3rd quartile", ls="--")
        plt.tight_layout()
        plt.show()

def invalidate(_=None):
    selected_estimators = [est for est, cb in estimators_checkboxes.items() if cb.value]
    plot_data(col_selection.value, configuration_selection.value, scenario_selection.value, selected_estimators)

# Connect all widgets
col_selection.observe(invalidate, names='value')
configuration_selection.observe(update_scenario_options, names='value')
scenario_selection.observe(invalidate, names='value')
for cb in estimators_checkboxes.values():
    cb.observe(invalidate, names='value')

# Display UI
controls_top = ipyw.HBox([col_selection, configuration_selection, scenario_selection])
controls_bottom = ipyw.VBox(list(estimators_checkboxes.values()))

display(ipyw.VBox([controls_top, controls_bottom, output]))

# Initial plot
invalidate()

<Figure size 640x480 with 0 Axes>